# Python 04 — Lists, Tuples, Sets, and Dictionaries

**Roadmap position:** Python → Core → Lists, tuples, sets, dictionaries.

Choose data structures by the operations your program needs—not by habit. This notebook uses realistic examples: ordered records, fixed values, duplicate detection, and fast lookup by ID.

## Outcome

You will be able to select and manipulate the four core collections, explain their trade-offs, and write small safe functions that use them.

## 1. Lists — ordered and mutable

A `list` stores an ordered sequence and can be changed. Use one when order matters or when you need to add/remove items. Common operations include `append`, indexing, iteration, and `pop`.

List lookup by value is normally linear: Python may need to inspect each item. Do not use a list as an ID index when you need frequent lookups.

In [1]:
pending_topics: list[str] = ['Python', 'SQL', 'HTTP']
pending_topics.append('Docker')
next_topic = pending_topics.pop(0)

print(f'Next topic: {next_topic}')
print(f'Remaining topics: {pending_topics}')

Next topic: Python
Remaining topics: ['SQL', 'HTTP', 'Docker']


## Your turn 1 — create a filtered list

Implement `valid_application_ids`. It receives a list of candidate IDs and returns a **new** list containing only non-empty IDs after surrounding whitespace is removed. Preserve the original order and do not modify the input list.

For this exercise, use a `for` loop, an initially empty output list, and `append`.

In [18]:
# YOUR TURN
def valid_application_ids(candidate_ids: list[str]) -> list[str]:
    valid_ids=[]
    for i in candidate_ids:
        cleaned_id=i.strip()
        if cleaned_id:
            valid_ids.append(cleaned_id)
         
    return valid_ids

In [19]:
# Checks — run after implementing valid_application_ids.
original_ids = [' app-1 ', '', 'app-2', '   ', 'app-3']
assert valid_application_ids(original_ids) == ['app-1', 'app-2', 'app-3']
assert original_ids == [' app-1 ', '', 'app-2', '   ', 'app-3']
assert valid_application_ids([]) == []
print('List-filter checks passed.')

List-filter checks passed.


## 2. Tuples — ordered and immutable

A `tuple` is ordered like a list but cannot be modified after creation. Use it for small, fixed groups of values, such as an immutable configuration pair or a function result with a stable positional meaning.

Immutability protects the tuple's structure; it does not make every object inside it immutable.

In [1]:
DEFAULT_REQUEST_TIMEOUTS: tuple[int, int] = (5, 30)  # connect seconds, read seconds
connect_timeout, read_timeout = DEFAULT_REQUEST_TIMEOUTS

print(f'Connect: {connect_timeout}s; read: {read_timeout}s')
# DEFAULT_REQUEST_TIMEOUTS[0] = 10  # TypeError: tuple does not support item assignment

Connect: 5s; read: 30s


## 3. Sets — unique values and fast membership

A `set` stores unique, hashable values. It is a strong choice for duplicate removal and membership checks such as “have we already processed this ID?” Order is not part of its contract, so do not use a set when output order is important.

Membership (`value in a_set`) is O(1) on average; list membership is O(n).

In [2]:
processed_document_ids: set[str] = {'doc-10', 'doc-11'}
incoming_document_id = 'doc-11'

if incoming_document_id in processed_document_ids:
    print('Duplicate document; skip processing.')
else:
    processed_document_ids.add(incoming_document_id)
    print('Document accepted for processing.')

Duplicate document; skip processing.


## Your turn 2 — identify duplicates

Implement `duplicate_ids`. It receives a list of strings and returns a set containing every ID that appears at least twice. A duplicated ID should occur once in the returned set.

Use two sets: one for IDs you have seen, and one for duplicates. Do not use `list.count`.

In [14]:
# YOUR TURN
def duplicate_ids(application_ids: list[str]) -> set[str]:
    ids = set()
    dupli_ids = set()
    for i in application_ids:
        if i in ids:
            dupli_ids.add(i)
        else:
            ids.add(i)
    return dupli_ids

In [15]:
# Checks — run after implementing duplicate_ids.
assert duplicate_ids([]) == set()
assert duplicate_ids(['a', 'b', 'c']) == set()
assert duplicate_ids(['a', 'b', 'a', 'a', 'b']) == {'a', 'b'}
print('Duplicate-ID checks passed.')

Duplicate-ID checks passed.


## 4. Dictionaries — keys mapped to values

A `dict` maps unique, hashable keys to values. Use it when you need a direct relationship: an application ID to its current status, a user ID to a profile, or a configuration key to a setting. Key lookup is O(1) on average.

Use `mapping[key]` when a missing key is an error. Use `mapping.get(key, default)` when absence is expected and you have a sensible fallback.

In [16]:
application_status_by_id: dict[str, str] = {
    'app-101': 'submitted',
    'app-102': 'interviewing',
}

known_status = application_status_by_id.get('app-102', 'unknown')
missing_status = application_status_by_id.get('app-999', 'unknown')
print(known_status)
print(missing_status)

interviewing
unknown


## Your turn 3 — count statuses

Implement `count_statuses`. It receives a list of status strings and returns a dictionary mapping each status to its count. For example, `['submitted', 'rejected', 'submitted']` becomes `{'submitted': 2, 'rejected': 1}`.

Start with an empty dictionary. For each status, read the current count safely, add one, then store it back. Do not use `collections.Counter` yet.

In [23]:
# YOUR TURN
def count_statuses(statuses: list[str]) -> dict[str, int]:
    status_count={}
    for i in statuses:
        if i not in status_count:
            status_count[i] = 1
        else:
            current_count = status_count[i]
            new_count = current_count + 1
            status_count[i]=new_count    
        
        
    return status_count    


In [24]:
# Checks — run after implementing count_statuses.
assert count_statuses([]) == {}
assert count_statuses(['submitted']) == {'submitted': 1}
assert count_statuses(['submitted', 'rejected', 'submitted']) == {
    'submitted': 2,
    'rejected': 1,
}
print('Status-count checks passed.')

Status-count checks passed.


## Debugging drill — do not mutate a collection while iterating it

The function below intends to remove blank IDs from a list, but it skips some invalid records.

1. Run it and compare the actual result with the expected result.
2. Explain why removal shifts later list positions.
3. Refactor it to return a new cleaned list instead of changing the input list while iterating.

**Interview question:** When would you choose a list, set, and dictionary for the same application domain?

In [29]:
# DEBUG ME
def remove_blank_ids(application_ids: list[str]) -> list[str]:
    cleaned_ids=[]
    for application_id in application_ids:
        if application_id.strip():
            cleaned_ids.append(application_id)
    return cleaned_ids


ids = ['app-1', '', '   ', 'app-2']
print(remove_blank_ids(ids))  # Expected: ['app-1', 'app-2']

['app-1', 'app-2']


## Exit interview check

Answer without running code:

1. When should you choose a list over a tuple?
2. Why can a set remove duplicates? Why does it not promise order?
3. When should you use `dict[key]` versus `dict.get(key, default)`?
4. Why is set/dictionary lookup O(1) on average?
5. Why is mutating a list while iterating it risky?

When finished, send me one completed exercise or debugging attempt at a time. I’ll review it without changing your work. Then we will proceed to **slicing**.